# Step 2 — Stateless FFT-peak HR Estimator

Simple per-window heart-rate estimator from PPG, per the paper's baseline spec.

**Pipeline (stateless — each 8 s window is estimated independently):**

1. **Bandpass-filter** each PPG channel with a 4th-order Butterworth (0.5–4 Hz, i.e. 30–240 BPM) using `filtfilt` for zero phase.
2. **Average** the two filtered PPG channels into one 1-D signal.
3. **Slice** into 8 s windows with 2 s shift (matches `BPM0` ground-truth cadence).
4. For each window, take the **zero-padded FFT** (N=8192 → ≈ 0.9 BPM bin resolution) and pick the dominant peak in [0.5, 4] Hz.
5. Convert peak frequency to BPM (×60) and compare against `BPM0`.

### Where the code lives

All pipeline functions are factored into [**`hr_estimator.py`**](hr_estimator.py) so this notebook — and any downstream step — can reuse them:

```python
from hr_estimator import (
    design_bandpass, bandpass_filter, average_channels,
    estimate_hr_window, window_spectrum,
    estimate_hr_trace, evaluate_recording, evaluate_all,
    HR_LO_HZ, HR_HI_HZ, N_FFT,
)
```

This notebook is the **sanity-check / visualization tour** over Step 2 — filter response, before/after-filter plots, FFT peak-pick diagnostics, per-recording and per-group MAE tables, and estimated-vs-true BPM traces.

> **Kernel:** select `Python (ppg-entropy)` in the top-right kernel picker before running.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.signal import freqz

from data_loader import (
    FS, WIN_SAMPLES, SHIFT,
    GROUP_COLORS, GROUP_LABELS,
    load_all_recordings,
)
from hr_estimator import (
    design_bandpass, bandpass_filter, average_channels,
    estimate_hr_window, window_spectrum,
    estimate_hr_trace, evaluate_recording, evaluate_all,
    HR_LO_HZ, HR_HI_HZ, N_FFT,
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 110


## 2.1 — Load recordings

**Expected output:**
- 22-line print from `load_all_recordings(verbose=True)` — same as Step 1
- Every line shows `ECG=yes` for T1 and `ECG=no` for T2/T3
- No assertion errors


In [ ]:
recordings = load_all_recordings(verbose=True)
print()
print(f'Loaded {len(recordings)} recordings.')


## 2.2 — Bandpass filter design sanity check

Plot the frequency response of the 4th-order Butterworth 0.5–4 Hz bandpass so we can visually confirm the passband and stopbands look right.

**Expected output:**
- Top panel (linear): flat ≈ 1.0 gain inside 0.5–4 Hz, rolloff to ≈ 0 outside
- Bottom panel (dB): passband ≈ 0 dB between 0.5 and 4 Hz
- −3 dB points should sit right at 0.5 and 4 Hz (dashed markers)
- No funky notches or overshoot in-band
- Filter order is 4 → rolloff around −24 dB/octave outside the band


In [ ]:
b, a = design_bandpass(fs=FS, lo=HR_LO_HZ, hi=HR_HI_HZ)
w, h = freqz(b, a, worN=4096, fs=FS)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 5), sharex=True)

ax1.plot(w, np.abs(h), color='steelblue')
ax1.axvline(HR_LO_HZ, color='grey', linestyle='--', lw=0.8)
ax1.axvline(HR_HI_HZ, color='grey', linestyle='--', lw=0.8)
ax1.set_ylabel('|H(f)|')
ax1.set_title(f'Butterworth bandpass — {HR_LO_HZ}–{HR_HI_HZ} Hz (order 4)')
ax1.grid(alpha=0.3)

ax2.plot(w, 20*np.log10(np.maximum(np.abs(h), 1e-6)), color='steelblue')
ax2.axvline(HR_LO_HZ, color='grey', linestyle='--', lw=0.8, label=f'{HR_LO_HZ} Hz (30 BPM)')
ax2.axvline(HR_HI_HZ, color='grey', linestyle='--', lw=0.8, label=f'{HR_HI_HZ} Hz (240 BPM)')
ax2.axhline(-3, color='red', linestyle=':', lw=0.8, label='−3 dB')
ax2.set_ylim(-80, 5)
ax2.set_xlim(0, 10)
ax2.set_ylabel('Gain (dB)')
ax2.set_xlabel('Frequency (Hz)')
ax2.legend(fontsize=8, loc='lower center')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()


## 2.3 — Raw vs bandpass-filtered PPG

Pick a representative recording from each group and show 30 s of raw PPG (ch1 + ch2) above and filtered-and-averaged PPG below. The filtered signal should look like a clean quasi-periodic AC waveform at the heart-rate frequency, motion-corrupted or not.

**Expected output:**
- Top panel: raw PPG, still has slow DC drift and motion bumps, maybe a visible pulse waveform
- Bottom panel: filtered-and-averaged PPG, centered around 0, oscillating at ~HR frequency
- T1 (treadmill, clean): filtered signal shows a clear near-periodic waveform, amplitude comparable throughout
- T3 (boxing, motion-heavy): filtered signal more irregular — motion artefacts in the 0.5–4 Hz band survive the filter
- If the filtered signal is flat or looks like pure noise, flag it (would indicate a broken filter or near-zero input)


In [ ]:
def plot_raw_vs_filtered(rec, duration_s=30):
    n   = int(duration_s * FS)
    t   = np.arange(n) / FS
    ppg = rec['ppg']
    ppg_filt = bandpass_filter(ppg, fs=FS)
    avg_filt = average_channels(ppg_filt)

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
    fig.suptitle(f"Recording {rec['rec_id']:02d} ({GROUP_LABELS[rec['group']]}) — first {duration_s} s",
                 fontsize=11)

    ax1.plot(t, ppg[0, :n], lw=0.6, color='royalblue', label='ch1 raw')
    ax1.plot(t, ppg[1, :n], lw=0.6, color='cornflowerblue', alpha=0.7, label='ch2 raw')
    ax1.set_ylabel('PPG raw')
    ax1.legend(fontsize=8, loc='upper right')

    ax2.plot(t, avg_filt[:n], lw=0.6, color=GROUP_COLORS[rec['group']])
    ax2.axhline(0, color='grey', lw=0.5)
    ax2.set_ylabel('PPG filtered + avg')
    ax2.set_xlabel('Time (s)')

    plt.tight_layout()
    plt.show()

# One representative recording per group
for rec_id in (1, 14, 16):      # T1 treadmill, T2 mixed, T3 boxing
    plot_raw_vs_filtered(next(r for r in recordings if r['rec_id'] == rec_id))


## 2.4 — Single-window FFT peak-pick diagnostic

For one 8 s window per group, plot the zero-padded power spectrum of the filtered-averaged PPG. Mark where the peak picker lands vs where the ground-truth BPM says the peak should be.

**Expected output:**
- Three spectra, one per representative recording/group (T1 / T2 / T3)
- Clear peak visible inside the 0.5–4 Hz band
- **Green dashed line** = picked frequency, **black solid line** = ground-truth BPM position
- If the two lines agree → estimator is working on that window
- If they disagree → estimator is locking onto a motion-artefact peak (often around running/boxing cadence, ~2–3 Hz)
- T3 boxing window should often show the "wrong peak" pattern — that is exactly the failure mode the paper is about


In [ ]:
def plot_fft_peak(rec, window_idx=10):
    """Show the FFT spectrum + picked peak + ground-truth BPM for one window."""
    ppg_filt = bandpass_filter(rec['ppg'], fs=FS)
    avg_filt = average_channels(ppg_filt)
    start    = window_idx * SHIFT
    x_win    = avg_filt[start : start + WIN_SAMPLES]

    freqs, power = window_spectrum(x_win, fs=FS)
    est_bpm  = estimate_hr_window(x_win, fs=FS)
    true_bpm = rec['bpm0'][window_idx]

    fig, ax = plt.subplots(figsize=(10, 3.2))
    band_mask = (freqs >= 0) & (freqs <= 6)
    ax.plot(freqs[band_mask], power[band_mask], color=GROUP_COLORS[rec['group']], lw=0.8)
    ax.axvspan(HR_LO_HZ, HR_HI_HZ, color='grey', alpha=0.1, label='HR band (0.5–4 Hz)')
    ax.axvline(est_bpm / 60.0,  color='green', linestyle='--', lw=1.2,
               label=f'Picked peak: {est_bpm:.1f} BPM')
    ax.axvline(true_bpm / 60.0, color='black', linestyle='-',  lw=1.2,
               label=f'Ground truth: {true_bpm:.1f} BPM')
    ax.set_xlim(0, 6)
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('|FFT|²')
    ax.set_title(f"Rec {rec['rec_id']:02d} ({GROUP_LABELS[rec['group']]}) — window {window_idx}  "
                 f"|est−true| = {abs(est_bpm - true_bpm):.1f} BPM")
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

# Mid-recording window for each representative recording
for rec_id in (9, 14, 16):
    rec = next(r for r in recordings if r['rec_id'] == rec_id)
    plot_fft_peak(rec, window_idx=min(40, len(rec['bpm0']) - 1))


## 2.5 — Run estimator over all 22 recordings; per-recording MAE

Run the full pipeline on every recording and compare against `BPM0` ground truth. Report mean absolute error (MAE) per recording.

**Expected output:**
- 22-line table, one row per recording: `rec_id | group | n_windows | MAE | median_err`
- **Median error** much smaller than MAE — indicates a heavy-tail failure distribution (the estimator is usually right, but catastrophically wrong on motion-corrupted windows)
- T1 treadmill: median error typically < 2 BPM; MAE varies (5–30 BPM) depending on how often the FFT locks onto running cadence instead of HR
- T3 boxing: MAE typically > 20 BPM — the paper's motivating failure mode
- Any recording with MAE < 1 BPM and median ≈ 0 is a very clean case; MAE > 30 BPM is a pathological failure


In [ ]:
results = evaluate_all(recordings)

hdr = f"{'Rec':>4}  {'Group':>6}  {'Windows':>8}  {'MAE (BPM)':>10}  {'Median err':>11}  {'Max err':>9}"
print(hdr)
print('-' * len(hdr))
for r in results:
    med = float(np.median(r['abs_error']))
    mx  = float(r['abs_error'].max())
    print(f"{r['rec_id']:>4}  {r['group']:>6}  {len(r['abs_error']):>8}  "
          f"{r['mae']:>10.2f}  {med:>11.2f}  {mx:>9.2f}")


## 2.6 — Per-recording MAE bar chart

Same 22-bar layout as Step 1's magnitude chart, now showing MAE in BPM. Colors reflect the three activity groups.

**Expected output:**
- 22 bars, coloured by group
- Wide variation across recordings, including within the same group (individual motion patterns matter)
- T3 boxing bars tend to dominate — they're the tallest on average
- A few very short bars (MAE < 5 BPM) scattered throughout where the FFT happened to lock the HR peak reliably
- Group means printed below the plot (more authoritative summary is in 2.8)


In [ ]:
rec_ids  = [r['rec_id'] for r in results]
maes     = [r['mae']    for r in results]
groups   = [r['group']  for r in results]
colors   = [GROUP_COLORS[g] for g in groups]

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(rec_ids, maes, color=colors, edgecolor='white', linewidth=0.5)
ax.set_xlabel('Recording ID')
ax.set_ylabel('MAE (BPM)')
ax.set_title('Per-recording HR-estimation MAE — stateless FFT-peak estimator')
ax.set_xticks(rec_ids)
ax.set_xticklabels([str(i) for i in rec_ids], fontsize=8)

handles = [mpatches.Patch(color=GROUP_COLORS[g], label=GROUP_LABELS[g]) for g in ('T1', 'T2', 'T3')]
ax.legend(handles=handles, loc='upper left', fontsize=9)

plt.tight_layout()
plt.show()

for grp in ('T1', 'T2', 'T3'):
    vals = [r['mae'] for r in results if r['group'] == grp]
    print(f'{GROUP_LABELS[grp]:30s}  n={len(vals):2d}  mean MAE: {np.mean(vals):6.2f}  (range {min(vals):5.2f}–{max(vals):5.2f})')


## 2.7 — Estimated vs true BPM traces

Plot estimated and ground-truth BPM versus window index (time) for one representative recording from each group. This is the clearest visual of where the estimator succeeds and fails across a recording.

**Expected output:**
- Three plots, one per group
- **Black line** = ground-truth BPM (`BPM0`), **coloured line** = estimator output
- T1 clean example (rec 9): the two lines should overlap almost perfectly
- T2 intermediate (rec 14): partial tracking, some excursions
- T3 failure case (rec 16): estimator diverges wildly — often flat-lines at the boxing cadence (~150–180 BPM) while true HR climbs/falls
- If the black line itself is flat or noisy, ground truth is suspect — flag it


In [ ]:
def plot_bpm_trace(result, rec):
    est  = result['est_bpm']
    true = result['true_bpm']
    t    = np.arange(len(est))  # window index (2 s each)

    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(t, true, color='black',   lw=1.2, label='Ground truth')
    ax.plot(t, est,  color=GROUP_COLORS[rec['group']], lw=1.0, alpha=0.85,
            label=f"Estimated (MAE = {result['mae']:.2f} BPM)")
    ax.set_xlabel('Window index (2 s each)')
    ax.set_ylabel('HR (BPM)')
    ax.set_title(f"Rec {rec['rec_id']:02d} — {GROUP_LABELS[rec['group']]}")
    ax.set_ylim(30, 240)
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

# Representative recording per group: clean T1, mid T2, pathological T3
for rec_id in (9, 14, 16):
    rec = next(r for r in recordings if r['rec_id'] == rec_id)
    res = next(r for r in results    if r['rec_id'] == rec_id)
    plot_bpm_trace(res, rec)


## 2.8 — Group-level MAE summary table

Pool all windows within each group and report MAE, median error, and a couple of quantiles so the heavy-tail story is visible in one place. This becomes part of the poster baseline.

**Expected output:**
- One row per group, plus a final pooled-all-22 row
- **Median error** noticeably smaller than MAE for every group — confirms the heavy-tail failure pattern
- **T1 mean MAE ≈ 13 BPM**, **T2 ≈ 15 BPM**, **T3 ≈ 24 BPM** (rough — will depend on the recordings). T3 should be the worst.
- p90 error gives a sense of how bad the bad windows are — likely ≫ 30 BPM for T3
- Numbers will be referenced in Step 3+ when we look for predictors (magnitude vs entropy) of per-window error


In [ ]:
def group_stats(err_arr, label):
    return dict(
        label  = label,
        n_win  = len(err_arr),
        mae    = float(err_arr.mean()),
        median = float(np.median(err_arr)),
        p90    = float(np.percentile(err_arr, 90)),
        max_   = float(err_arr.max()),
    )

rows = []
for grp in ('T1', 'T2', 'T3'):
    err = np.concatenate([r['abs_error'] for r in results if r['group'] == grp])
    rows.append(group_stats(err, GROUP_LABELS[grp]))
rows.append(group_stats(np.concatenate([r['abs_error'] for r in results]), 'ALL (pooled)'))

hdr = f"{'Group':30s}  {'n_win':>6}  {'MAE':>8}  {'Median':>8}  {'p90':>8}  {'Max':>8}"
print(hdr)
print('-' * len(hdr))
for r in rows:
    print(f"{r['label']:30s}  {r['n_win']:>6}  {r['mae']:>8.2f}  {r['median']:>8.2f}  {r['p90']:>8.2f}  {r['max_']:>8.2f}")
